In [10]:
import json
import re


def parse_float_list(text):
    return [float(x.strip()) for x in text.split(",")]


folder = "../results/vae_no_warmup/"
input_file = folder + "vae_no_warmup_short.txt"
output_file = folder + "vae_no_warmup_short.json"

with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

# Warmup weights belong to the NEXT epoch
warmup_weights = {
    int(epoch) + 1: float(weight)
    for epoch, weight in re.findall(
        r"Epoch\s+(\d+):\s+KL warmup weight set to\s+([-\deE+.]+)",
        text,
    )
}

blocks = re.split(r"=+\s*Epoch\s+(\d+)\s*=+", text)

epochs = []

for i in range(1, len(blocks), 2):
    epoch = int(blocks[i])
    block = blocks[i + 1]

    data = {"epoch": epoch}

    # ----------------------------------------
    # Main metrics
    # ----------------------------------------

    patterns = {
        "train_loss": r"Train loss:\s*([-\deE+.]+)",
        "val_nll": r"NLL on val set:\s*([-\deE+.]+)",
        "test_nll": r"NLL on test set:\s*([-\deE+.]+)",
        "posterior_val_nll": r"Posterior NLL on val:\s*([-\deE+.]+)",
        "posterior_train_nll": r"Posterior NLL on train:\s*([-\deE+.]+)",
        "prior_train_nll": r"Prior NLL on train:\s*([-\deE+.]+)",
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, block)
        if match:
            data[key] = float(match.group(1))

    # ----------------------------------------
    # Layer KL
    # ----------------------------------------

    match = re.search(
        r"Layered KL per layer:\s*\[([^\]]+)\]",
        block,
    )
    if match:
        data["layered_kl"] = parse_float_list(match.group(1))

    # ----------------------------------------
    # Lambda (scalar)
    # ----------------------------------------

    match = re.search(
        r"lambda_\s*per layer:\s*([-\deE+.]+)",
        block,
    )
    if match:
        data["lambda"] = float(match.group(1))

    # ----------------------------------------
    # Reconstruction / KL loss
    # ----------------------------------------

    match = re.search(
        r"reconstruction=([-\deE+.]+)\s+kl_loss=([-\deE+.]+)",
        block,
    )
    if match:
        data["reconstruction"] = float(match.group(1))
        data["kl_loss"] = float(match.group(2))

    # ----------------------------------------
    # Scrabbled NLL
    # ----------------------------------------

    match = re.search(
        r"Scrabbled NLL on train:\s*([-\deE+.]+),\s*"
        r"Scrabbled NLL on val:\s*([-\deE+.]+)",
        block,
    )
    if match:
        data["scrabbled_train_nll"] = float(match.group(1))
        data["scrabbled_val_nll"] = float(match.group(2))

    # ----------------------------------------
    # KL warmup weight
    # ----------------------------------------

    if epoch in warmup_weights:
        data["kl_warmup_weight"] = warmup_weights[epoch]

    epochs.append(data)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(epochs, f, indent=4)

print(f"Parsed {len(epochs)} epochs")
print(f"Saved to {output_file}")

Parsed 27 epochs
Saved to ../results/vae_no_warmup/vae_no_warmup_short.json
